In [1]:
# import
import pandas as pd
import os
import numpy as np

In [ ]:
def safe_read_csv(filepath: str) -> pd.DataFrame:
    """
    여러 인코딩을 시도하여 CSV 파일을 안전하게 읽는 함수

    Parameters:
        filepath (str): CSV 파일 경로

    Returns:
        pd.DataFrame: 성공적으로 로드된 DataFrame
    """
    # 사용할 인코딩 후보 리스트
    encodings = ["cp949", "utf-8-sig", "utf-8", "euc-kr"]
    last_error = None  # 마지막으로 발생한 오류 저장

    # 후보 인코딩을 순서대로 시도
    for enc in encodings:
        try:
            # 주어진 인코딩으로 CSV 읽기
            df = pd.read_csv(filepath, encoding=enc)

            # 읽은 파일이 비어 있으면 오류 발생
            if df.empty:
                raise ValueError("CSV 파일이 비어 있습니다.")

            # 정상적으로 읽었으면 DataFrame 반환
            return df

        except Exception as e:
            # 실패하면 오류 기록 후 다음 인코딩 시도
            last_error = e
            continue

    # 모든 인코딩 시도 후에도 실패한 경우 예외 발생
    raise ValueError(f"CSV 로드 실패: {last_error}")


In [ ]:
""" 리팩토링 진행 완료한 코드"""

# 파일 내 첫 번째 컬럼명을 확인하고, 시군구 컬럼(법정동 정보)을 표준화, 정규화하는 함수
def integration_sgg_col(
    filepath: str
)-> pd.DataFrame:
    """
    주어진 CSV 파일의 첫 번째 컬럼명을 확인하여
    시군구 컬럼을 표준화·정규화한 뒤 result.csv로 저장

    Parameters:
        filepath (str): 입력 CSV 파일 경로

    Returns:
        pd.DataFrame: 시군구 컬럼 정제 및 표준화가 완료된 DataFrame
    """
    # 1) 인코딩 오류 방지하며 파일 불러오기
    df = safe_read_csv(filepath)
    first_col = df.columns[0] # 첫 번째 컬럼명
    
    # 2) 케이스 분기 
    if first_col == "시군구별(1)": # 법정동 컬럼 2개
        df = combine_region_columns(df) # 시군구 컬럼 2개 하나로 합치기(함수 제작 필요)
        df = replace_abbreviated_sido_names(df, column="시군구별", col_cnt=2) # 시도 치환
        df = merge_subdistricts_to_city(df, region_col="SGG_NAME") # 하위 행정구역 통합
    elif first_col =="시군구별": # 법정동 컬럼 1개
        df = replace_abbreviated_sido_names(df, column=first_col, col_cnt=1) # 시도 치환
        df = create_full_region_column(df, column="SGG_NAME") # 시군구_전체 컬럼 생성
    else: # 둘다 아닐때 데이터 초기 전처리 잘못되었으므로 오류 발생 내용 추가
        raise ValueError(
            f"지원하지 않는 첫 컬럼명: '{first_col}'. 허용: '시군구별(1)', '시군구별'"
        )
    
    df = merge_gunwigun(df, region_col="SGG_NAME") # 군위군 데이터 통합
    df = remove_target_regions(df, region_col="SGG_NAME") # 특례시 하위 행정구역 제거
    df = remove_only_sido_rows(df, region_col="SGG_NAME") # 시도만 있는 행 제거 
    df = remove_seoul_gyeonggi_incheon(df)
    df = check_missing_regions(df, region_col="SGG_NAME")
    validate_and_save_dataframe(df)
    
    # 이거 함수 구현
    df.to_csv("./result.csv", encoding="utf-8-sig", index=False)
    return df

In [ ]:
def data_cleansing_folder(
    folder_path: str, 
    output_folder: str
)-> None:
    """
    폴더 내 모든 .csv 파일을 읽어 데이터 정제 후 output_folder에 저장

    Parameters:
        folder_path (str): .csv 파일이 있는 폴더 경로
        output_folder (str): 결과 저장할 폴더 경로
    """
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".csv"):
            file_path = os.path.join(folder_path, filename)
            print(f"📄 처리 중: {filename}")

            try:
                # 시군구 컬럼 표준화, 정규화
                df = integration_sgg_col(file_path)

                # 결과 파일명 정의
                output_filename = filename.replace(".csv", "_정리.csv")
                output_path = os.path.join(output_folder, output_filename)

                # 저장
                df.to_csv(output_path, index=False, encoding="utf-8-sig")
                print(f"✅ 저장 완료 → {output_path}")

            except Exception as e:
                print(f"❌ 오류 발생 ({filename}): {e}")

In [ ]:
# 수정 필요
input_path = "./data"
# 수정 필요
output_path = "./result"


# 나중에 하나
# 폴더 안 파일들 전체 데이터 정제
data_cleansing_folder(input_path, output_path)